# Could the neighbour galaxy have helped MoM-BH\*-1 form?

*A separate test, not used in the research note in this repository (see the README).*

**The idea being tested (direct collapse):** normally, a big cloud of gas cools and breaks up into many ordinary stars. Molecular hydrogen (H₂) is what lets primordial gas cool. Ultraviolet light in the **Lyman–Werner band** (about 912–1,110 Å) destroys H₂. If a nearby galaxy bathes the cloud in enough of this light, the gas can't cool, can't fragment, and may collapse into a single enormous object that becomes a black hole of about 10⁴–10⁶ solar masses. The discovery paper suggests MoM-BH\*-1's neighbour might have played this role.

**What we'll do:**
1. Measure the neighbour's ultraviolet brightness from its spectrum.
2. Turn that into how much Lyman–Werner light reaches MoM-BH\*-1 (it weakens with the square of the distance).
3. Compare with the level theory says direct collapse needs.

**The unit:** the intensity is measured as **J21**, in units of 10⁻²¹ erg s⁻¹ cm⁻² Hz⁻¹ sr⁻¹. Theoretical estimates of the direct-collapse threshold vary between studies, from roughly **100 to 10,000**, with **about 1,000** a commonly used value.

**How to use:** run the grey code cells from top to bottom with **Shift + Enter**.

*Data: Naidu et al., Zenodo record 15059215 (CC BY 4.0).*

## Step 1 — Download both spectra

In [ ]:
!wget -q -O spec_150135.fits "https://zenodo.org/records/15059215/files/mom-uds01-v3_prism-clear_5224_150135.spec.fits?download=1"
!wget -q -O spec_948311.fits "https://zenodo.org/records/15059215/files/mom-uds01-v3_prism-clear_5224_948311.spec.fits?download=1"

import os
for name in ["spec_150135.fits", "spec_948311.fits"]:
    size_kb = os.path.getsize(name) / 1024 if os.path.exists(name) else 0
    print(f"{name}: {size_kb:.0f} KB ->", "OK" if size_kb > 300 else "download failed, run this cell again in a few minutes")

## Step 2 — How far apart are they?

The two objects are about 64 kpc apart. Remember it's the distance **across the sky** (projected). The real distance can only be the same or larger.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy.cosmology import Planck18
from astropy import units as u

Z = 7.7569   # both objects are at the same redshift (the paper finds a difference below 0.01)


def load_spectrum(name):
    '''Return wavelength (µm), flux and error (µJy), and the sky position (RA, Dec).'''
    with fits.open(name) as f:
        ext = "SPEC1D" if "SPEC1D" in f else 1
        tab = Table(f[ext].data)
        ra = dec = None
        for hdu in f:
            if "SRCRA" in hdu.header:
                ra, dec = hdu.header["SRCRA"], hdu.header["SRCDEC"]
                break
    cols = {c.lower(): c for c in tab.colnames}
    err_col = cols.get("err", cols.get("full_err"))
    wave = np.array(tab[cols["wave"]], dtype=float)
    flux = np.array(tab[cols["flux"]], dtype=float)
    err = np.array(tab[err_col], dtype=float)
    if np.nanmedian(wave) > 100:
        wave = wave / 1e4
    good = np.isfinite(wave) & np.isfinite(flux) & np.isfinite(err) & (err > 0)
    return wave[good], flux[good], err[good], ra, dec


wave_bh, flux_bh, err_bh, ra_bh, dec_bh = load_spectrum("spec_150135.fits")   # MoM-BH*-1
wave_n, flux_n, err_n, ra_n, dec_n = load_spectrum("spec_948311.fits")        # the neighbour galaxy

# if a file has no stored position, use positions measured from the images
if ra_bh is None: ra_bh, dec_bh = 34.39748407592, -5.13511630627
if ra_n is None: ra_n, dec_n = 34.4009232, -5.1342402

sep = SkyCoord(ra_bh * u.deg, dec_bh * u.deg).separation(SkyCoord(ra_n * u.deg, dec_n * u.deg)).to(u.arcsec)
distance = (sep * Planck18.kpc_proper_per_arcmin(Z)).to(u.kpc)
print(f"Separation on the sky: {sep:.2f}  ->  distance: {distance:.1f} (minimum, because it's projected)")

## Step 3 — Measure the neighbour's ultraviolet brightness

We average the neighbour's brightness between **1,400 and 1,600 Å** (rest frame), the standard "UV" wavelength for galaxies.

We can't measure the Lyman–Werner band (about 1,000 Å) directly. At this redshift, the neutral hydrogen between galaxies absorbs almost all light bluer than 1,216 Å on its way to us. So we'll use the 1,500 Å brightness as a stand-in, which is fair for young, hot stars to within about a factor of 2.

The cell also prints the UV **absolute magnitude** (more negative means brighter) and a rough **star formation rate** (Kennicutt 1998). The plot shows both objects: the neighbour is bright in the UV where the black hole star is almost dark.

In [ ]:
UV_WINDOW = (1400, 1600)   # rest-frame Å

rest_n = wave_n * 1e4 / (1 + Z)
inside = (rest_n >= UV_WINDOW[0]) & (rest_n <= UV_WINDOW[1])
f_uv = flux_n[inside].mean() * 1000                                    # nJy
f_uv_err = np.sqrt(np.sum(err_n[inside] ** 2)) / inside.sum() * 1000
print(f"Neighbour's UV brightness (1400-1600 Å rest): {f_uv:.1f} ± {f_uv_err:.1f} nJy  ({inside.sum()} pixels)")

D_L = Planck18.luminosity_distance(Z)
L_uv = (4 * np.pi * D_L**2 * (f_uv * u.nJy) / (1 + Z)).to(u.erg / u.s / u.Hz)
m_uv = -2.5 * np.log10(f_uv * 1e-9 / 3631)                              # apparent AB magnitude
M_uv = m_uv - Planck18.distmod(Z).value + 2.5 * np.log10(1 + Z)         # absolute UV magnitude
sfr = 1.4e-28 * L_uv.value                                              # Kennicutt (1998)
print(f"UV luminosity:          {L_uv:.2e}")
print(f"Absolute UV magnitude:  {M_uv:.1f}")
print(f"Rough star formation rate: {sfr:.0f} solar masses of new stars per year")

# plot both spectra in the rest frame
rest_bh = wave_bh * 1e4 / (1 + Z)
fig, ax = plt.subplots(figsize=(10, 4))
ax.step(rest_n, flux_n * 1000, where="mid", color="tab:blue", lw=1, label="neighbour galaxy")
ax.step(rest_bh, flux_bh * 1000, where="mid", color="black", lw=1, label="MoM-BH*-1")
ax.axvspan(*UV_WINDOW, color="tab:blue", alpha=0.2, label="UV window (1400-1600 Å)")
ax.axvline(1216, color="gray", ls="--", lw=0.8)
ax.text(1216, ax.get_ylim()[1] * 0.95, "Lyman-α ", ha="right", va="top", fontsize=9, color="gray")
ax.set_xlim(1000, 6000)
ax.set_xlabel("Rest-frame wavelength (Å)")
ax.set_ylabel("Brightness (nJy)")
ax.legend(loc="upper center", fontsize=9)
plt.tight_layout()
plt.show()

## Step 4 — How much Lyman–Werner light reaches MoM-BH\*-1?

For light spreading out in all directions from a source of luminosity L, the intensity at distance d is:

**J = L / (16 π² d²)**

(the 16π² comes from spreading the light over a sphere, 4πd², and then per steradian, another 4π).

The cell then compares your J21 with three threshold values (100, 1,000 and 10,000). It also works out how close the neighbour would have needed to be to reach each one. Because J falls as 1/d², the needed distance is your distance × √(J21 / threshold).

In [ ]:
LW_FACTOR = 1.0     # assume the galaxy is about as bright at ~1000 Å as at 1500 Å (true to within ~2x)
L_lw = LW_FACTOR * L_uv
J21_UNIT = 1e-21 * u.erg / u.s / u.cm**2 / u.Hz / u.sr


def J21_at(d):
    '''Lyman-Werner intensity (in J21 units) at distance d from the neighbour.'''
    return (L_lw / (16 * np.pi**2 * d.to(u.cm) ** 2) / u.sr / J21_UNIT).decompose().value


J21 = J21_at(distance)
print(f"Lyman-Werner intensity reaching MoM-BH*-1 today: J21 = {J21:.1f}\n")
for J_crit in [100, 1000, 10000]:
    d_needed = distance * np.sqrt(J21 / J_crit)
    verdict = "ABOVE" if J21 >= J_crit else "below"
    print(f"threshold J21 = {J_crit:>6,}: the neighbour's light is {verdict} it  "
          f"(the neighbour would need to be within {d_needed.value:.1f} kpc)")

## Step 5 — Plot: the neighbour's UV light versus distance

The blue line shows how strong the neighbour's Lyman–Werner light is at each distance. The orange band is the range of theoretical thresholds for direct collapse. The red dot is where MoM-BH\*-1 sits today. The plot is saved as **MoM-BH1_neighbour_test.png**.

In [ ]:
d_grid = np.logspace(0, np.log10(300), 300) * u.kpc

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(d_grid.value, J21_at(d_grid), color="tab:blue", lw=2, label="neighbour's Lyman-Werner light")
ax.axhspan(100, 10000, color="tab:orange", alpha=0.2, label="theoretical thresholds for direct collapse")
ax.axhline(1000, color="tab:orange", ls="--", lw=1)
ax.plot(distance.value, J21, "o", color="tab:red", ms=9, label=f"MoM-BH*-1 today ({distance.value:.0f} kpc)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Distance from the neighbour galaxy (kpc)")
ax.set_ylabel("Lyman-Werner intensity, J21")
ax.set_title("Could the neighbour's UV light have enabled direct collapse?")
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
fig.savefig("MoM-BH1_neighbour_test.png", dpi=150)
plt.show()

## What your result means

- **If J21 is far below the whole orange band:** the neighbour, as it is now and where it is now, gives far too little UV light to have triggered direct collapse. Direct collapse would then need either a closer or brighter source in the past (see the "within ... kpc" numbers), or another source of UV. Otherwise the black hole formed some other way: for example, a lighter seed that grew very fast, or a black hole built up in a dense star cluster.
- **If J21 is inside or above the band:** this supports the direct-collapse idea the paper suggests for MoM-BH\*-1.
- **Does MoM-BH\*-1 need a heavy seed at all?** The research note in this repository finds that its black hole is probably below a few × 10⁴ M☉. A black hole that light could also have grown from a much smaller seed, so this test does not decide how it formed.
- Either way, you've put a **number** on an idea the paper only mentions in words. Baggen et al. (2026) did similar calculations for much closer companions (within about 5 kpc) of other little red dots, which you can compare with.

**Caveats** (mention these if you write it up):
- **Timing:** we see the neighbour as it is now. When the black hole formed, the neighbour may have been closer, brighter (for example during a burst of star formation), or not yet shining.
- **Distance:** 64 kpc is the minimum distance, so today's J21 is an **upper limit**.
- **The ~1,000 Å brightness** is estimated from 1,500 Å; this could be off by about a factor of 2.
- **The threshold itself** is uncertain by a factor of about 100 between theoretical studies. That's the biggest uncertainty in the whole test.
- MoM-BH\*-1's own faint host galaxy, and other nearby galaxies, add a little UV light that isn't counted here.